# Silver: parse road status into sections
Reads the latest raw page from `brp.bronze.road_status_raw` and turns its two HTML tables
(Virginia, North Carolina) into a clean table, `brp.silver.road_sections_status`: one row per parkway section per day.

**Steps**
1. **Settings:** imports, table names, and the silver schema
2. **Load:** get the latest bronze snapshot
3. **Helpers + self-tests:** clean text, find the state, detect headers, parse mileposts, classify status
4. **Extract:** copy every row of both tables, unfiltered, so nothing can be lost unnoticed
5. **Transform:** sort each row into one bucket (section, header, single-milepost, or unparsed) and build clean section records
6. **Quality checks:** stop if any row is unaccounted for or unparsed, a range is invalid, or the sections leave a gap/overlap larger than `GAP_TOLERANCE_MI` (currently 0.2 mi) between milepost 0 and `PARKWAY_LENGTH_MI` (469.1); warn on unknown statuses
7. **Write:** replace this day's rows in silver (safe to re-run)
8. **Review:** sections and miles by state + status, by state, by status, and grand total

**Status rules** (checked in order, first match wins): "single lane"/"delay" → reduced · "closed" → closed ·
starts with "open"/"ungated" → open · otherwise → unknown.
Columns are read by **position**, because header wording differs between the two tables.
Single-milepost rows (one number, not a range) are set aside on purpose; the gap check proves no road miles are lost.

In [0]:
# ---------------------------------------------------------------------------
# SETTING AND IMPORTS
# ---------------------------------------------------------------------------

# Standard library (built into Python, no install needed)
import re   # regular expressions: pattern matching on text

# Third-party (installed packages)
import pandas as pd     # tabular data in plain Python; handy for previewing records before Spark
from bs4 import BeautifulSoup   # parses the raw HTML so we can find tables, rows and cells
from pyspark.sql.types import ( # Spark data types, used to define the table structure below
    StructType, StructField, DateType, StringType, IntegerType, DoubleType, BooleanType,TimestampType
)


# ---------------------------------------------------------------------------
# TABLE NAMES
# Full Unity Catalog names, in the form catalog.schema.table.
# Defined once here so the rest of the notebook never hard-codes them.
# ---------------------------------------------------------------------------

# Bronze = raw input: the NPS road status page saved exactly as downloaded.
# This notebook reads from it.
BRONZE_TABLE = "brp.bronze.road_status_raw"

# Silver = cleaned output: one row per parkway section per run, with proper
# types and normalized status values. This notebook writes to it.
SILVER_TABLE = "brp.silver.road_sections_status"


# ---------------------------------------------------------------------------
# SILVER TABLE SCHEMA
# The exact column names, data types and order of the silver table.
# Passing this to Spark (e.g. spark.createDataFrame(records, SILVER_SCHEMA))
# means Spark uses these types instead of guessing them, so every run writes
# the same structure. All columns are nullable (the default).
# ---------------------------------------------------------------------------

SILVER_SCHEMA = StructType([
        # --- When the data was captured ---
    StructField("run_date",      DateType()),       # date this pipeline ran (Eastern)
    StructField("fetched_at",    TimestampType()),  # exact download time (UTC); links to the bronze row
    StructField("nps_as_of",     StringType()),     # NPS's own "Road status as of..." time, kept as text

    # --- Where the section is ---
    StructField("state",         StringType()),   # "VA" or "NC"
    StructField("section_order", IntegerType()),  # position on the page (1, 2, 3...), keeps original order
    StructField("mp_start",      DoubleType()),   # starting milepost, e.g. 0.0
    StructField("mp_end",        DoubleType()),   # ending milepost, e.g. 2.2
    StructField("length_mi",     DoubleType()),   # section length in miles (mp_end - mp_start)
    StructField("crossroads",    StringType()),   # roads or landmarks at each end of the section

    # --- What the status is ---
    StructField("status_raw",    StringType()),   # status text exactly as NPS wrote it
    StructField("notes",         StringType()),   # any extra remarks NPS added for the section
    StructField("status",        StringType()),   # cleaned category: open / reduced / closed / unknown
    StructField("gated",         BooleanType()),  # True if the section is gated (status doesn't start with "Ungated")
])

In [0]:
# ---------------------------------------------------------------------------
# LOAD THE LATEST RAW PAGE FROM BRONZE
# The bronze table stores one row per saved download (several per day are
# possible). We only need the newest one: sort by the exact download time
# (fetched_at), newest first, and keep just the top row.
# ---------------------------------------------------------------------------

snapshot = spark.sql(f"""
    SELECT run_date, fetched_at, raw_html   -- capture day, exact download time, and the page HTML
    FROM {BRONZE_TABLE}
    ORDER BY fetched_at DESC                -- newest download first 
    LIMIT 1                                 -- keep only that one
""").first()
# .first() returns that single row as a Spark Row object.

# If the bronze table is empty, it returns None instead.
assert snapshot is not None, f"{BRONZE_TABLE} is empty: run the bronze notebook first"

# Pull the values out of the row by column name. Both are stamped on every silver row.
run_date   = snapshot["run_date"]      # a Python date, e.g. 2026-11-13 (Eastern calendar day)
fetched_at = snapshot["fetched_at"]    # a Python datetime, the exact download time (UTC);
                                       # links each silver row back to its bronze download

# Turn the raw HTML text into a searchable structure. The helper cells below
# use `soup` to find the "as of" timestamp, the state tables, rows and cells.
# "html.parser" is Python's built-in parser, so nothing extra needs installing.
soup = BeautifulSoup(snapshot["raw_html"], "html.parser")

print("Processing download from:", fetched_at, f"(run date {run_date})")

In [0]:
# ---------------------------------------------------------------------------
# HELPER FUNCTIONS
# Small, single-purpose functions used by the extract and transform cells.
# Each one turns messy text from the NPS page into a clean value for one or
# more silver columns. They're tested at the bottom of this cell.
# ---------------------------------------------------------------------------


def clean(text):
    """Collapse line breaks and repeated spaces into single spaces."""
    # Text pulled from HTML is often messy: get_text() can return things like
    # "0.0 - 2.2\n\n" or "Open   with\tdelays", because of how the page is laid out.
    #   text or ""   -> treat None as an empty string, so this never crashes
    #   \s+          -> any run of whitespace (spaces, tabs, line breaks)...
    #   " "          -> ...gets replaced with one single space
    #   .strip()     -> remove spaces left at the very start and end
    # Example: "  Open\n   with delays " -> "Open with delays"
    return re.sub(r"\s+", " ", text or "").strip()


# Pattern for a milepost range ANYWHERE in a cell, e.g. "0.0 - 2.2", "291.8 – 305.1",
# or "MP 216.9 - 229.6" (text before the numbers is allowed).
#   (\d+(?:\.\d+)?)    -> a number, with optional decimals (291 or 291.8); captured as group(1)
#   \s*[-–—]\s*        -> a dash with optional spaces around it; accepts a hyphen (-),
#                         en dash (–) or em dash (—), since web pages mix them
#   (\d+(?:\.\d+)?)    -> the second number; captured as group(2)
# No ^ at the start, and .search() below, so the range can appear anywhere.
# Compiled once here so it isn't rebuilt for every row.
MP_RANGE = re.compile(r"(\d+(?:\.\d+)?)\s*[-–—]\s*(\d+(?:\.\d+)?)")


def parse_mileposts(text):
    """Return (start, end) as numbers, or None if no milepost range is found."""
    # Look for a milepost range anywhere in the cell.
    match = MP_RANGE.search(text)

    if not match:
        # No range found. The transform cell puts this row in the "unparsed"
        # bucket, where it is reported and fails the quality checks,
        # so it can never disappear silently.
        return None

    # The pattern captured both numbers as text; convert them to real numbers.
    start, end = float(match.group(1)), float(match.group(2))

    # Always return (smaller, larger), so a range written backwards on the
    # page, like "305.1 - 291.8", still comes out as (291.8, 305.1).
    return min(start, end), max(start, end)


def is_header(cells):
    """True if this row is a table's column-title row."""
    # Each table starts with a title row whose first cell reads
    # "Parkway Mileposts by Section" (with a line break in the HTML).
    # Identifying it explicitly means a header is recognized on purpose,
    # instead of being treated as "a row that failed to parse".
    #   bool(cells) -> False for an empty row, so cells[0] can't crash
    return bool(cells) and "mileposts by section" in cells[0].lower()


def state_for_table(tbl):
    """Find the 'Virginia/North Carolina Sections of Parkway' label above a table."""
    # The state isn't inside the table; it's in a label just above it (a <p>
    # paragraph), such as "Virginia Sections of Parkway". So starting from the
    # table, look backwards up the page for the nearest text containing
    # "Sections of Parkway". re.I makes the search ignore upper/lower case.
    label = tbl.find_previous(string=re.compile(r"Sections of Parkway", re.I))

    # If no label was found, find_previous returns None. Swap that for ""
    # so .lower() works, then lowercase it to make the checks below simple.
    label = (label or "").lower()

    if "virginia" in label:
        return "VA"
    if "north carolina" in label:
        return "NC"

    # Neither state found: this isn't one of the two road status tables
    # (e.g. a layout or unrelated table). The extract cell skips it.
    return None


def classify_status(status_raw):
    """Map the NPS status text to open / reduced / closed / unknown (order matters)."""
    # Lowercase once so the checks below ignore capitalization.
    s = status_raw.lower()

    # The checks run top to bottom, and the first match wins. Order matters
    # because one status can contain several keywords: "Open with sporadic
    # single lane closures" contains "open", "single lane" AND "closures".

    # 1. Partly open: the road is usable but has lane closures or delays.
    #    Checked first so these aren't mislabeled "closed" or "open".
    if "single lane" in s or "delay" in s:
        return "reduced"

    # 2. Closed: the word "closed" appears and none of the rule-1 words did.
    if "closed" in s:
        return "closed"

    # 3. Fully open: the text starts with "Open" or "Ungated" and none of the
    #    earlier rules matched (e.g. "Open", "Ungated*").
    if s.startswith("open") or s.startswith("ungated"):
        return "open"

    # 4. Anything else: wording we haven't planned for, or empty text.
    #    These rows are still saved, and the quality checks print a warning
    #    so someone can add a rule for the new wording.
    return "unknown"


# ---------------------------------------------------------------------------
# SELF-TESTS
# Each line feeds a sample input to a helper and checks the answer. If any
# helper gives the wrong result, the notebook stops here with an error
# naming the failed test, before any data is parsed or saved.
# ---------------------------------------------------------------------------

# Text cleaning: line breaks and extra spaces collapse; None is safe
assert clean("  Open\n   with delays ") == "Open with delays", "Self-test: clean whitespace"
assert clean(None) == "", "Self-test: clean(None) -> empty string"

# Milepost parsing: a real range is split into numbers; a header returns None
assert parse_mileposts("0.0 - 2.2") == (0.0, 2.2), "Self-test: basic milepost range"
assert parse_mileposts("Parkway Mileposts by Section") is None, "Self-test: header row should return None"

# Milepost parsing: text before the numbers is allowed (the missed-NC-row case)
assert parse_mileposts("MP 216.9 - 229.6") == (216.9, 229.6), "Self-test: range with prefix text"

# Milepost parsing: en dash and a backwards range
assert parse_mileposts("291.8 – 305.1") == (291.8, 305.1), "Self-test: en dash"
assert parse_mileposts("305.1 - 291.8") == (291.8, 305.1), "Self-test: backwards range is reordered"

# Header detection: the title row is a header; a data row is not; an empty row is safe
assert is_header(["Parkway Mileposts by Section", "Crossroads", "Status", "Notes"]), "Self-test: header row detected"
assert not is_header(["0.0 - 2.2", "Rockfish Gap", "Open", ""]), "Self-test: data row is not a header"
assert not is_header([]), "Self-test: empty row is not a header"

# Status classification: one test for each category, plus the tricky mixed wording
assert classify_status("Open") == "open", "Self-test: 'Open' -> open"
assert classify_status("Ungated*") == "open", "Self-test: 'Ungated*' -> open"
assert classify_status("Closed") == "closed", "Self-test: 'Closed' -> closed"
assert classify_status("Open with sporadic single lane closures") == "reduced", "Self-test: open + lane closures -> reduced"
assert classify_status("Ungated with sporadic single lane closures") == "reduced", "Self-test: ungated + lane closures -> reduced"
assert classify_status("") == "unknown", "Self-test: empty text -> unknown"

print("Helper self-tests passed")

In [0]:
# ---------------------------------------------------------------------------
# EXTRACT EVERY ROW OF THE ROAD STATUS TABLES, UNFILTERED
# The page has one table for Virginia and one for North Carolina.
# Copy every row of both into a DataFrame exactly as written: headers,
# odd rows, everything. Nothing is interpreted or skipped here, so no
# row can disappear without notice. Cleaning happens in the next cell.
# ---------------------------------------------------------------------------

raw_rows = []   # one dictionary per table row; becomes the DataFrame below

# enumerate() numbers the tables 0, 1, 2... in page order.
# select("table") finds every <table> tag (CSS selector syntax).
for table_index, tbl in enumerate(soup.select("table")):

    # The state label ("Virginia Sections of Parkway") sits in a <p> just
    # ABOVE the table, so state_for_table() looks backwards from the table.
    # Tables without that label aren't road status tables: skip them.
    state = state_for_table(tbl)
    if state is None:
        continue

    # Every <tr> (table row), in order, numbered from 0
    for row_index, tr in enumerate(tbl.select("tr")):

        # Text of every cell in the row. "td, th" means <td> OR <th>.
        # get_text(" ") joins pieces split by <br /> with a space;
        # clean() collapses extra spaces and line breaks.
        cells = [clean(cell.get_text(" ")) for cell in tr.select("td, th")]

        raw_rows.append({
            "state":       state,         # "VA" or "NC"
            "table_index": table_index,   # which table on the page
            "row_index":   row_index,     # position within the table (0 = first row)
            "cell_count":  len(cells),    # how many cells this row has
            "cells":       cells,         # list of cell text, in order
        })

# Turn the list of dictionaries into a table:
# each dictionary becomes a row, each key becomes a column.
raw_df = pd.DataFrame(raw_rows)

# Quick summary: total rows and rows per state
print(len(raw_df), "rows extracted:", raw_df["state"].value_counts().to_dict())
display(raw_df)

In [0]:
# ---------------------------------------------------------------------------
# TRANSFORM: TURN EXTRACTED ROWS INTO CLEAN SECTION RECORDS
# Every row from raw_df lands in exactly ONE bucket:
#   headers  -> the column-title row of each table (expected, skipped)
#   singles  -> a row whose first cell holds ONE milepost, not a range
#               (recognized on purpose and set aside; not a section)
#   unparsed -> anything else that isn't a usable section (fails the checks)
#   records  -> a parsed parkway section (goes to silver)
# Columns are read by POSITION because header wording differs between
# the two tables: 0 = mileposts, 1 = crossroads, 2 = status, 3 = notes.
# ---------------------------------------------------------------------------

# The page's own timestamp, e.g. "Road status as of 12:00 P.M, Thursday, November 12, 2026."
# Different from run_date and fetched_at: this is when NPS last updated the page.
# re.sub with re.I removes the prefix whatever its capitalization, matching the search.
as_of_text = soup.find(string=re.compile(r"Road status as of", re.I))
nps_as_of = (
    re.sub(r"Road status as of", "", clean(as_of_text), flags=re.I).strip(" .")
    if as_of_text else None
)

NUMBER = re.compile(r"\d+(?:\.\d+)?")   # any number, e.g. 63 or 63.5

records  = []   # parsed sections (dictionaries)
headers  = []   # header rows
singles  = []   # single-milepost rows (set aside on purpose)
unparsed = []   # rows that didn't fit any expected pattern
section_order = 0   # running number, north to south, across both tables

# to_dict("records") gives one plain dictionary per row, e.g.
# {"state": "VA", "table_index": 0, "row_index": 1, "cell_count": 4, "cells": [...]}
for row in raw_df.to_dict("records"):
    cells = row["cells"]

    # Bucket 1: header row ("Parkway Mileposts by Section", "Crossroads", ...)
    if is_header(cells):
        headers.append(row)
        continue

    # Try to read a milepost range ("0.0 - 2.2") from the first cell
    mileposts = parse_mileposts(cells[0]) if cells else None

    # Bucket 2: exactly one number in the first cell, e.g. "63.5": a single
    # milepost, not a section. Set aside on purpose; the continuity check
    # (Check 8) proves no road miles are missing because of this.
    if mileposts is None and cells and len(NUMBER.findall(cells[0])) == 1:
        singles.append(row)
        continue

    # Bucket 3: anything else that isn't a usable section
    if mileposts is None or len(cells) < 3:
        unparsed.append(row)
        continue

    # Bucket 4: a real section. Name the columns by position.
    mp_start, mp_end = mileposts
    status_raw = cells[2]
    section_order += 1

    # Keys are in the SAME ORDER as SILVER_SCHEMA. sections_df takes its column
    # order from here, and pandas -> Spark can match columns by position.
    records.append({
        "run_date":      run_date,                                    # Eastern calendar day of the download
        "fetched_at":    fetched_at,                                  # exact download time (UTC); links back to the bronze row
        "nps_as_of":     nps_as_of,                                   # when NPS last updated the page
        "state":         row["state"],
        "section_order": section_order,
        "mp_start":      mp_start,
        "mp_end":        mp_end,
        "length_mi":     round(mp_end - mp_start, 2),
        "crossroads":    cells[1],
        "status_raw":    status_raw,                                  # exactly as NPS wrote it
        "notes":         cells[3] if len(cells) > 3 else "",          # 4th column may be missing
        "status":        classify_status(status_raw),                 # our classification
        "gated":         not status_raw.lower().startswith("ungated"),
    })

# The clean table, ready for quality checks and writing
sections_df = pd.DataFrame(records)

# Accounting summary: the four buckets should add up to the number extracted
print("NPS status as of:", nps_as_of)
print("Download being processed:", fetched_at)
print(f"Extracted: {len(raw_df)} | Sections: {len(records)} | Headers: {len(headers)} | "
      f"Single-milepost: {len(singles)} | Unparsed: {len(unparsed)}")

for row in singles:
    print(f"  SINGLE   {row['state']} table {row['table_index']} row {row['row_index']}: {row['cells']}")
for row in unparsed:
    print(f"  UNPARSED {row['state']} table {row['table_index']} row {row['row_index']}: {row['cells']}")

display(sections_df)

In [0]:
# ---------------------------------------------------------------------------
# QUALITY CHECKS
# Run before anything is written to silver.
#   FAIL checks (assert) -> stop the notebook (and a scheduled job) so bad
#                            data is never saved
#   WARN checks (print)  -> data is saved, but something needs a human look
# Completeness rule: the sections must cover the parkway end to end, with
# no gap or overlap larger than GAP_TOLERANCE_MI between neighbours.
# ---------------------------------------------------------------------------

ALLOWED_STATUSES = {"open", "reduced", "closed", "unknown"}
PARKWAY_LENGTH_MI = 469.1      # official length of the parkway
STATE_LINE_MP = 217            # the parkway crosses from VA into NC near milepost 217
GAP_TOLERANCE_MI = 0.2         # largest allowed gap/overlap between neighbouring sections

# ---- FAIL checks ----------------------------------------------------------

# 1. Accounting: every extracted row landed in exactly one bucket
assert len(records) + len(headers) + len(singles) + len(unparsed) == len(raw_df), \
    "Check 1: row accounting mismatch (rows were lost or double-counted)"

# 2. No unexpected rows: single-milepost rows are set aside on purpose,
#    but anything else that didn't parse must be investigated
assert not unparsed, \
    f"Check 2: {len(unparsed)} rows could not be parsed; see the UNPARSED list above"

# 3. We found sections from both states
assert len(records) > 0, "Check 3: no sections parsed; has the page layout changed?"
states_found = set(sections_df["state"])
assert states_found == {"VA", "NC"}, f"Check 3: expected VA and NC, found {states_found}"

# 4. Every section has a valid milepost range with a positive length
for r in records:
    assert 0 <= r["mp_start"] < r["mp_end"] <= PARKWAY_LENGTH_MI + 1, \
        f"Check 4: bad milepost range {r['state']} {r['mp_start']}-{r['mp_end']}"

# 5. States match geography: VA sections north of the state line, NC south of it
for r in records:
    if r["state"] == "VA":
        assert r["mp_end"] <= STATE_LINE_MP + 1, f"Check 5: VA section too far south: {r['mp_start']}-{r['mp_end']}"
    else:
        assert r["mp_start"] >= STATE_LINE_MP - 1, f"Check 5: NC section too far north: {r['mp_start']}-{r['mp_end']}"

# 6. Only the expected status categories are used
bad_statuses = set(sections_df["status"]) - ALLOWED_STATUSES
assert not bad_statuses, f"Check 6: unexpected status values {bad_statuses}"

# 7. Coverage of the ends: the first section starts at milepost 0 and the
#    last one reaches the end of the parkway
ordered = sorted(records, key=lambda r: r["mp_start"])
assert ordered[0]["mp_start"] <= GAP_TOLERANCE_MI, \
    f"Check 7: first section starts at {ordered[0]['mp_start']}, not 0"
assert ordered[-1]["mp_end"] >= PARKWAY_LENGTH_MI - GAP_TOLERANCE_MI, \
    f"Check 7: last section ends at {ordered[-1]['mp_end']}, not {PARKWAY_LENGTH_MI}"

# 8. Continuity: each section starts where the previous one ends
#    (within GAP_TOLERANCE_MI). Every problem is listed before failing.
problems = []
for prev, curr in zip(ordered, ordered[1:]):
    difference = round(curr["mp_start"] - prev["mp_end"], 2)
    if abs(difference) > GAP_TOLERANCE_MI:
        kind = "GAP" if difference > 0 else "OVERLAP"
        problems.append(f"{kind} of {abs(difference)} mi between "
                        f"{prev['mp_start']}-{prev['mp_end']} and {curr['mp_start']}-{curr['mp_end']}")
for p in problems:
    print("Check 8 PROBLEM:", p)
assert not problems, f"Check 8: {len(problems)} gaps/overlaps larger than {GAP_TOLERANCE_MI} mi (listed above)"

total_miles = sections_df["length_mi"].sum()
print(f"FAIL checks 1-8 passed: {len(records)} sections cover {total_miles:.1f} of {PARKWAY_LENGTH_MI} miles "
      f"with no gap or overlap over {GAP_TOLERANCE_MI} mi")

# ---- WARN checks ----------------------------------------------------------

# 9. Unknown statuses: NPS used wording our rules don't cover yet
for r in records:
    if r["status"] == "unknown":
        print(f"Check 9 WARNING: unknown status at {r['state']} {r['mp_start']}-{r['mp_end']}: {r['status_raw']!r}")

# 10. NPS "as of" timestamp found on the page
if nps_as_of is None:
    print("Check 10 WARNING: 'Road status as of' text not found on the page")

# 11. Single-milepost rows set aside (for information)
if singles:
    print(f"Check 11 NOTE: {len(singles)} single-milepost rows set aside (listed in the transform output)")

print("Quality checks complete (see any WARNING/NOTE lines above)")

In [0]:
# ---------------------------------------------------------------------------
# WRITE TO SILVER (SAFE TO RE-RUN)
# Only runs if every FAIL check above passed.
# Replaces just THIS DOWNLOAD's rows (matched on fetched_at) in the silver table:
#   - rerunning silver on the same download overwrites its rows instead of
#     duplicating them
#   - other downloads, including earlier ones from the same day, are left
#     untouched
# ---------------------------------------------------------------------------

# Put each record's values in the same order as the schema's columns.
# Values are looked up BY NAME (r[field.name]), so the dictionary's key order
# doesn't matter, and a key missing from the records stops here with a
# KeyError instead of quietly becoming NULL.
rows = [tuple(r[field.name] for field in SILVER_SCHEMA.fields) for r in records]

# Build a Spark DataFrame (Spark is what writes Delta tables).
silver_df = spark.createDataFrame(rows, schema=SILVER_SCHEMA)

(silver_df.write
    .mode("overwrite")                                                # overwrite...
    .option("replaceWhere", f"fetched_at = TIMESTAMP '{fetched_at}'") # ...but ONLY this download's rows
    .saveAsTable(SILVER_TABLE))

print(f"Wrote {len(rows)} sections to {SILVER_TABLE} for download {fetched_at} (run date {run_date})")

In [0]:
%sql
-- ---------------------------------------------------------------------------
-- REVIEW: SECTIONS AND MILES BY STATE AND STATUS, WITH TOTALS
-- Summarizes the latest day at four levels in one result:
--   1. by state + status   2. by state   3. by status   4. grand total
-- CUBE(state, status) computes every combination of the two columns at once,
-- instead of writing four separate queries.
-- Quick check: the grand total should be about 469.1 miles, and the two
-- state totals should add up to it.
-- ---------------------------------------------------------------------------

SELECT
  -- In total rows, the column being totalled comes back as NULL.
  -- GROUPING(column) returns 1 when that column is being totalled, 0 otherwise,
  -- so it tells us which level each row belongs to.
  -- The numbers in the labels make ORDER BY sort the levels in this order.
  CASE
    WHEN GROUPING(state) = 1 AND GROUPING(status) = 1 THEN '4. Grand total'
    WHEN GROUPING(state) = 1                          THEN '3. Total by status'
    WHEN GROUPING(status) = 1                         THEN '2. Total by state'
    ELSE                                                   '1. By state + status'
  END                                    AS level,

  -- COALESCE(value, 'ALL') -> use the value, or 'ALL' if it is NULL,
  -- so total rows read "ALL" instead of an empty cell
  COALESCE(state,  'ALL')                AS state,
  COALESCE(status, 'ALL')                AS status,

  COUNT(*)                               AS sections,   -- number of sections
  ROUND(SUM(length_mi), 1)               AS miles       -- total miles, 1 decimal place

FROM brp.silver.road_sections_status

-- Latest day only: the subquery finds the newest run_date in the table
WHERE run_date = (SELECT MAX(run_date) FROM brp.silver.road_sections_status)

GROUP BY CUBE(state, status)

-- Levels 1 -> 4; within each level VA before NC (DESC), then status A -> Z
ORDER BY level, state DESC, status